# AI-Orb – Stimmserver auf Google Colab

Dieses Notebook macht aus einer kostenlosen Colab-Sitzung einen Sprachserver
für den Orb. Er nimmt Text entgegen und gibt gesprochene Sprache zurück –
mit **deiner eigenen Stimme**, geklont aus einer kurzen Aufnahme.

## Zwei Modelle, umschaltbar

| | **XTTS-v2** | **CSM-1b** (Sesame) |
|---|---|---|
| Deutsch | nativ trainiert | kaum – *„likely won't do well"* laut Sesame |
| Stimme klonen | aus ~6 s Referenz | aus Referenz im Gesprächskontext |
| Stärke | verlässlich, klare Aussprache | sehr lebendige Sprechdynamik |

**Ehrlich vorweg:** Die Stimmen *Maya* und *Miles* aus der Sesame-Demo sind
**nicht** im offenen Modell enthalten. Sesame hat nur das Basismodell ohne
Stimmen veröffentlicht. Was hier entsteht, ist die *Sprechdynamik* dieser
Modellfamilie mit **deiner** Stimme – nicht Miles selbst.

## So gehst du vor

1. **Laufzeit → Laufzeittyp ändern → T4 GPU** wählen (kostenlos).
2. Zellen der Reihe nach ausführen (Umschalt+Enter).
3. In Zelle 3 deine Stimmprobe hochladen (10–30 Sekunden, ruhig gesprochen).
4. Zelle 5 gibt dir eine Adresse wie `https://xxxx.trycloudflare.com`.
   Die trägst du beim Orb ein – dafür gibt es `STIMME-VERBINDEN.bat`.

**Colab-Grenzen, die du kennen solltest:** Die Sitzung endet nach etwa
90 Minuten ohne Aktivität und spätestens nach 12 Stunden. Danach ist die
Adresse eine andere. Der Orb merkt das von selbst und fällt still auf seine
eingebaute Stimme zurück – er verstummt nicht.

## 1. Prüfen, ob eine GPU da ist

In [ ]:
import subprocess, sys

try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total',
                                   '--format=csv,noheader'], text=True).strip())
    GPU = True
except Exception:
    GPU = False

if not GPU:
    print('\n*** KEINE GPU AKTIV ***')
    print('Oben im Menue: Laufzeit -> Laufzeittyp aendern -> T4 GPU -> Speichern.')
    print('Ohne GPU laeuft es zwar, aber ein Satz dauert dann Minuten statt Sekunden.')
else:
    print('\nGPU ist da. Weiter mit Zelle 2.')

## 2. Pakete installieren

Dauert beim ersten Mal etwa 3–5 Minuten. Warnungen über Versionskonflikte
sind hier normal und stören nicht.

In [ ]:
%%capture --no-stderr
# XTTS-v2 (deutschfaehig) ueber die gepflegte Abspaltung von Coqui.
!pip install -q coqui-tts==0.25.1
# Server und oeffentlicher Zugang
!pip install -q fastapi uvicorn nest_asyncio pydantic soundfile
# CSM-1b (Sesame) - Transformers bringt es seit 4.52 mit
!pip install -q --upgrade "transformers>=4.52" accelerate
# Der Tunnel, der die Adresse nach draussen gibt
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
print('fertig')

In [ ]:
print('Installation abgeschlossen. Weiter mit Zelle 3.')

## 3. Deine Stimmprobe hochladen

**Was eine gute Probe ausmacht:**

* 10–30 Sekunden, WAV oder MP3
* nur du, kein Hintergrundgeräusch, keine Musik
* ruhig und deutlich gesprochen – so, wie der Orb später klingen soll
* ein zusammenhängender Text ist besser als einzelne Wörter

Der Orb bringt dafür `STIMME-AUFNEHMEN.bat` mit; du kannst aber jede
beliebige Aufnahme nehmen. **Ohne Probe** geht es auch weiter – dann nimmt
XTTS eine mitgelieferte Standardstimme.

In [ ]:
import os, glob

os.makedirs('/content/stimme', exist_ok=True)
REFERENZ = None

try:
    from google.colab import files
    print('Waehle deine Aufnahme aus (Abbrechen = ohne Referenz weitermachen):')
    hochgeladen = files.upload()
    for name in hochgeladen:
        ziel = f'/content/stimme/{name}'
        os.replace(name, ziel)
        REFERENZ = ziel
except Exception as fehler:
    print(f'Kein Upload moeglich ({fehler}) - es geht ohne Referenz weiter.')

if REFERENZ is None:
    vorhanden = glob.glob('/content/stimme/*')
    REFERENZ = vorhanden[0] if vorhanden else None

print()
print('Referenzstimme:', REFERENZ or 'keine - es wird die Standardstimme benutzt')

## 4. Modelle laden

XTTS-v2 wird immer geladen (das ist der deutschfähige Weg). CSM-1b wird
zusätzlich geladen, wenn genug Speicher da ist – du kannst später im
laufenden Betrieb zwischen beiden umschalten und selbst vergleichen.

In [ ]:
import torch, os

GERAET = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Rechenwerk:', GERAET)

# --- XTTS-v2 ---------------------------------------------------------
os.environ['COQUI_TOS_AGREED'] = '1'   # nichtkommerzielle Lizenz bestaetigen
XTTS = None
try:
    from TTS.api import TTS as CoquiTTS
    XTTS = CoquiTTS('tts_models/multilingual/multi-dataset/xtts_v2').to(GERAET)
    print('XTTS-v2 bereit (Deutsch moeglich).')
except Exception as fehler:
    print('XTTS-v2 liess sich NICHT laden:', fehler)

# --- CSM-1b ----------------------------------------------------------
CSM = None
CSM_PROZESSOR = None
try:
    from transformers import CsmForConditionalGeneration, AutoProcessor
    CSM_PROZESSOR = AutoProcessor.from_pretrained('sesame/csm-1b')
    CSM = CsmForConditionalGeneration.from_pretrained(
        'sesame/csm-1b',
        torch_dtype=torch.float16 if GERAET == 'cuda' else torch.float32,
    ).to(GERAET)
    print('CSM-1b bereit (englisch; deutsch nur als Versuch).')
except Exception as fehler:
    print('CSM-1b liess sich nicht laden:', fehler)
    print('Das ist kein Beinbruch - XTTS traegt den Betrieb.')

if XTTS is None and CSM is None:
    raise SystemExit('Kein Modell geladen - bitte Zelle 2 noch einmal ausfuehren.')

## 5. Server starten

Danach erscheint eine Adresse. **Diese Adresse brauchst du für den Orb.**

Die Schnittstelle ist genau die, die der Orb schon erwartet:
`POST /tts` mit `{"text": "..."}` – zurück kommt eine WAV-Datei.

In [ ]:
import io, re, subprocess, threading, time, wave

import nest_asyncio, numpy as np, soundfile as sf, uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse, Response
from pydantic import BaseModel

nest_asyncio.apply()
app = FastAPI(title='AI-Orb Stimmserver')

# Welches Modell gerade spricht. Umschaltbar im Betrieb ueber /modell.
AKTIV = 'xtts' if XTTS is not None else 'csm'


class Anfrage(BaseModel):
    text: str
    speaker: int | str | None = 0
    speed: float | None = 1.0
    sprache: str | None = 'de'
    modell: str | None = None      # 'xtts' | 'csm' | None = das aktive


def _als_wav(daten, rate: int) -> bytes:
    """Macht aus Zahlenreihen eine WAV-Datei im Speicher."""
    puffer = io.BytesIO()
    reihe = np.asarray(daten, dtype=np.float32)
    spitze = float(np.max(np.abs(reihe))) if reihe.size else 0.0
    if spitze > 1.0:
        reihe = reihe / spitze
    sf.write(puffer, reihe, rate, format='WAV', subtype='PCM_16')
    return puffer.getvalue()


def _sprich_xtts(text: str, sprache: str, tempo: float) -> bytes:
    kwargs = {'text': text, 'language': (sprache or 'de')[:2], 'speed': tempo}
    if REFERENZ:
        kwargs['speaker_wav'] = REFERENZ
    else:
        # Ohne Referenz eine mitgelieferte Stimme nehmen.
        kwargs['speaker'] = 'Claribel Dervla'
    welle = XTTS.tts(**kwargs)
    return _als_wav(welle, 24000)


def _sprich_csm(text: str, sprecher: int) -> bytes:
    eingabe = CSM_PROZESSOR(f'[{sprecher}]{text}', add_special_tokens=True).to(CSM.device)
    with torch.no_grad():
        audio = CSM.generate(**eingabe, output_audio=True)
    welle = audio[0].to(torch.float32).cpu().numpy()
    return _als_wav(welle, 24000)


@app.get('/health')
def health():
    return {
        'status': 'ok',
        'aktiv': AKTIV,
        'xtts': XTTS is not None,
        'csm': CSM is not None,
        'referenz': bool(REFERENZ),
    }


@app.post('/modell')
def modell_waehlen(wahl: dict):
    """Im Betrieb umschalten: {"modell": "csm"} oder {"modell": "xtts"}."""
    global AKTIV
    gewuenscht = str(wahl.get('modell', '')).lower().strip()
    if gewuenscht == 'xtts' and XTTS is not None:
        AKTIV = 'xtts'
    elif gewuenscht == 'csm' and CSM is not None:
        AKTIV = 'csm'
    else:
        return JSONResponse({'fehler': f'{gewuenscht!r} steht nicht bereit.'}, 400)
    return {'aktiv': AKTIV}


@app.post('/tts')
def tts(anfrage: Anfrage):
    """Die Schnittstelle, die der Orb benutzt."""
    text = (anfrage.text or '').strip()
    if not text:
        return JSONResponse({'fehler': 'Es fehlt der Text.'}, 400)
    # Zu lange Eingaben abschneiden - sonst wartet der Orb minutenlang.
    text = text[:600]

    gewaehlt = (anfrage.modell or AKTIV).lower()
    tempo = float(anfrage.speed or 1.0)
    tempo = min(1.6, max(0.6, tempo))

    begonnen = time.perf_counter()
    try:
        if gewaehlt == 'csm' and CSM is not None:
            sprecher = anfrage.speaker if isinstance(anfrage.speaker, int) else 0
            audio = _sprich_csm(text, sprecher)
        elif XTTS is not None:
            audio = _sprich_xtts(text, anfrage.sprache or 'de', tempo)
        else:
            return JSONResponse({'fehler': 'Kein Modell bereit.'}, 503)
    except Exception as fehler:
        return JSONResponse({'fehler': f'{type(fehler).__name__}: {fehler}'}, 500)

    dauer = time.perf_counter() - begonnen
    print(f'[{gewaehlt}] {dauer:5.1f}s  {text[:60]!r}')
    return Response(content=audio, media_type='audio/wav')


# --- Server im Hintergrund starten -----------------------------------
def _server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error')

threading.Thread(target=_server, daemon=True).start()
time.sleep(4)

# --- Tunnel nach draussen --------------------------------------------
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

ADRESSE = None
for zeile in tunnel.stdout:
    treffer = re.search(r'https://[-\w]+\.trycloudflare\.com', zeile)
    if treffer:
        ADRESSE = treffer.group(0)
        break

print()
print('=' * 62)
if ADRESSE:
    print('  Der Stimmserver laeuft.')
    print()
    print('  ADRESSE FUER DEN ORB:')
    print(f'      {ADRESSE}')
    print()
    print('  So verbindest du ihn:')
    print('    Fuehre im AI-Orb-Ordner STIMME-VERBINDEN.bat aus und')
    print('    fuege die Adresse ein. Dann den Orb neu starten.')
    print()
    print(f'  Aktives Modell: {AKTIV}   (umschalten: siehe Zelle 6)')
else:
    print('  Der Tunnel kam nicht zustande. Fuehre diese Zelle noch einmal aus.')
print('=' * 62)
print()
print('DIESE ZELLE MUSS LAUFEN BLEIBEN. Schliesse den Tab nicht.')

## 6. Anhören und vergleichen

Hier hörst du beide Modelle mit demselben deutschen Satz. **Genau dafür ist
diese Zelle da:** Du entscheidest nach Gehör, nicht nach meiner Einschätzung.

In [ ]:
from IPython.display import Audio, display

SATZ = 'Guten Morgen. Ich habe drei neue Nachrichten gefunden. Soll ich sie dir vorlesen?'

if XTTS is not None:
    print('--- XTTS-v2 (deutschfaehig) ---')
    display(Audio(_sprich_xtts(SATZ, 'de', 1.0), autoplay=False))

if CSM is not None:
    print('--- CSM-1b auf Deutsch (Sesame sagt selbst: schwach) ---')
    display(Audio(_sprich_csm(SATZ, 0), autoplay=False))
    print('--- CSM-1b auf Englisch (dafuer ist es gebaut) ---')
    display(Audio(_sprich_csm(
        'Good morning. I found three new messages. Would you like me to read them?', 0), autoplay=False))

print()
print('Umschalten im laufenden Betrieb:')
print("    import requests; requests.post(f'{ADRESSE}/modell', json={'modell': 'csm'})")

## 7. Wach halten (freiwillig)

Colab beendet eine Sitzung nach etwa 90 Minuten ohne Aktivität. Diese Zelle
meldet sich alle paar Minuten selbst und schiebt das hinaus – bis zur
harten Grenze von 12 Stunden.

*Hinweis: Dauerbetrieb ist auf der Gratisstufe nicht vorgesehen. Für den
gelegentlichen Gebrauch ist das in Ordnung; wer den Orb täglich damit
sprechen lassen will, ist mit Colab Pro oder einem kleinen Server besser
bedient.*

In [ ]:
import time, urllib.request

print('Haelt die Sitzung wach. Zum Beenden: Stopp-Knopf links.')
while True:
    try:
        with urllib.request.urlopen('http://localhost:8000/health', timeout=5) as antwort:
            antwort.read()
        print('.', end='', flush=True)
    except Exception as fehler:
        print(f'\nServer antwortet nicht: {fehler}')
    time.sleep(240)